<a href="https://www.kaggle.com/code/alexvmt/terainet-inference-example?scriptVersionId=335791988" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# TeraiNet inference example

Follow [mewc-predict](https://github.com/zaandahl/mewc-predict/blob/main/src/mewc_predict.py) for the general procedure

## Setup

### Imports

Follow [mewc-flow](https://github.com/zaandahl/mewc-flow/blob/main/requirements.txt) for the key package versions

In [ ]:
!pip install keras==3.3.3 kimm==0.2.5 tensorflow==2.16.1

In [ ]:
import os
import random
import shutil
from pathlib import Path

import pandas as pd
import tensorflow as tf
import yaml
from keras import saving

### Utilities

In [ ]:
def copy_random_images(src_dir, dst_dir, n, seed=42):
    """
    Copies n random images from src_dir to dst_dir.

    Args:
        src_dir (str): Source directory containing images.
        dst_dir (str): Destination directory to copy images into.
        n (int): Number of images to copy.
        seed (int): Random seed for reproducibility.
    """
    # Ensure target directory exists
    os.makedirs(dst_dir, exist_ok=True)

    # List all files in the source directory
    all_files = [f for f in os.listdir(src_dir) if os.path.isfile(os.path.join(src_dir, f))]

    # Check if n is greater than available files
    if n > len(all_files):
        raise ValueError(
            f"Requested {n} images, but only {len(all_files)} available in source directory."
        )

    # Randomly sample n files
    random.seed(seed)
    selected_files = random.sample(all_files, n)

    # Copy each file to the target directory
    for filename in selected_files:
        src_path = os.path.join(src_dir, filename)
        dst_path = os.path.join(dst_dir, filename)
        shutil.copy2(src_path, dst_path)

    print(f"Copied {n} random images from '{src_dir}' to '{dst_dir}'.")

## Prepare images

In [ ]:
copy_random_images("../input/preprocess-images/terainet_images/test2/class_1", "../images", 10)

In [ ]:
img_generator = tf.keras.preprocessing.image_dataset_from_directory(
    "../images", labels=None, label_mode=None, batch_size=8, image_size=(224, 224), shuffle=False
)

## Predict

In [ ]:
model = saving.load_model("../input/train-and-evaluate-terainet/model.keras", compile=False)

In [ ]:
preds = model.predict(img_generator)

In [ ]:
preds

## Post-processing

Follow [mewc-predict](https://github.com/zaandahl/mewc-predict/blob/main/src/mewc_predict.py)

In [ ]:
with open("../input/train-and-evaluate-terainet/class_list.yaml", "r") as file:
    class_map = yaml.safe_load(file)
class_map

In [ ]:
inv_class = {v: k for k, v in class_map.items()}
inv_class

In [ ]:
file_paths = img_generator.file_paths
filenames = list(map(lambda x: Path(x).name, file_paths))
labels = list(map(lambda x: Path(x).parent.name, file_paths))

In [ ]:
class_ids = sorted(inv_class.values())
class_names = [class_map.get(i, i) for i in class_ids]
pred_df = pd.DataFrame(preds, columns=class_ids)
pred_df.head()

In [ ]:
file_series = pd.Series(filenames)
label_series = pd.Series(labels)
pred_df.insert(0, "filename", file_series, True)
pred_df.insert(1, "label", label_series, True)
pred_df = pd.melt(
    pred_df,
    id_vars=["filename", "label"],
    value_vars=class_ids,
    var_name="class_id",
    value_name="prob",
)
pred_df["class_name"] = pred_df["class_id"].replace(class_map)
pred_df["class_rank"] = pred_df.groupby("filename")["prob"].rank("average", ascending=False)
pred_df.head(10)

In [ ]:
pred_df = pred_df[pred_df["class_rank"] == 1.0]
pred_df = pred_df.drop(["label", "class_rank"], axis=1)
pred_df